# 期現套利 HBT：跨日留倉 Runner

這個 Notebook 是可調參數的薄 runner；底層邏輯放在同資料夾與 `future_spot/arbitrage/` 的 Python 模組。預設執行 2026 年 1–7 月單次連續回測：未平倉部位會帶入下一交易日，不在隔日篩選名單的舊合約也會保留，到期日不得殘倉或自動換月。既有 event NPZ 會直接重用。

In [1]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()
TEST_ROOT = CURRENT_DIR if (CURRENT_DIR / 'backtest_config.py').exists() else CURRENT_DIR / 'future_spot' / 'test'
if not TEST_ROOT.exists():
    raise FileNotFoundError('Open this notebook from future_spot/test or the repository root')
PROJECT_ROOT = TEST_ROOT.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
for path in (TEST_ROOT, PROJECT_ROOT, WORKSPACE_ROOT, PROJECT_ROOT / 'scripts'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from IPython.display import Image, display
from backtest_config import default_notebook_args
from backtest_pipeline import run_backtest_pipeline
from report_tables import build_report_tables
from report_plots import save_report_plots

## 1. 參數

In [2]:
args = default_notebook_args(
    start_date='2026-01-01',
    end_date='2026-07-31',
    carry_positions=True,
    total_capital=50_000_000.0,
    futures_margin_rate=0.20,
    spot_equity_rate=0.40,
    leverage=False,  # False：期貨與現股皆按 100% 自有資金
    future_order_latency_ms=1.0,
    future_response_latency_ms=1.0,
    future_feed_latency_offset_ms=0.0,
    spot_order_latency_ms=1.0,
    spot_response_latency_ms=35.0,
    spot_feed_latency_offset_ms=0.0,
    low_memory_reports=True,  # 報表前釋放大型 frames，改由 CSV 分批彙總
    report_mode='summary',  # 不建立 failure windows / 逐筆 capital 明細
    report_chunk_rows=25_000,
    continue_on_error=True,  # 保留已知缺資料日；下方仍會獨立檢查到期日殘倉
    post_first_feed_wait='spot',
    post_first_feed_timeout_ms=5000.0,
    rebuild_hbt_results=False,  # manifest 失效時仍會自動重跑 HBT
)

# 已確認的 5 筆到期殘倉例外；只略過稽核，不從交易與績效資料刪除。
EXPIRY_AUDIT_EXCLUDED_RUN_KEYS = {
    '2026-04-15::1802_KUFD6',
    '2026-07-15::2408_CYFG6',
    '2026-07-15::5371_NMFG6',
    '2026-07-15::2340_FYFG6',
    '2026-07-15::6257_MQFG6',
}

args.output_dir

PosixPath('/home/zoufuc/hftbacktest/future_spot/output/hbt_daily_full_market_20260101_20260731_future_order_1ms_response_1ms_feed_0ms_spot_order_1ms_response_35ms_feed_0ms')

## 2. 執行完整回測與檢查跨日持倉

In [3]:
artifacts = run_backtest_pipeline(args)
display(artifacts.frame('summary').head(20))
display(artifacts.frame('entry_exit_index').head(20))

carry_status = artifacts.frame('position_carry_status')
display(carry_status.loc[carry_status['universe_source'].ne('selected')].head(30))
all_expiry_violations = carry_status.loc[carry_status['status'].eq('expiry_position_remaining')].copy()
skipped_expiry_violations = all_expiry_violations.loc[
    all_expiry_violations['run_key'].isin(EXPIRY_AUDIT_EXCLUDED_RUN_KEYS)
]
expiry_violations = all_expiry_violations.loc[
    ~all_expiry_violations['run_key'].isin(EXPIRY_AUDIT_EXCLUDED_RUN_KEYS)
]
if not skipped_expiry_violations.empty:
    print('以下已確認的到期殘倉例外不納入本次稽核：')
    display(skipped_expiry_violations)
if not expiry_violations.empty:
    display(expiry_violations)
    raise RuntimeError(f'排除已確認例外後，期貨到期日仍有殘倉：{len(expiry_violations)} 筆')
print('跨日留倉稽核通過：未發現未列入例外清單的到期日殘倉。')

,trade_date,run_key,pair_name,spot_symbol,future_symbol,rows,filled_pairs,second_leg_failures,flatten_count,realized_pnl,...,post_first_feed_poll_ns,spot_order_latency_ns,future_order_latency_ns,spot_response_latency_ns,future_response_latency_ns,spot_feed_latency_offset_ns,future_feed_latency_offset_ns,strategy_engine,scan_calls,python_decisions
0,2026-01-08,2026-01-08::1303_CAFA6,1303_CAFA6,1303,CAFA6,0,0,0,0,0.0,...,10000000,1000000,1000000,35000000,1000000,0,0,numba,265,264
1,2026-01-08,2026-01-08::2002_CBFA6,2002_CBFA6,2002,CBFA6,0,0,0,0,0.0,...,10000000,1000000,1000000,35000000,1000000,0,0,numba,265,264
2,2026-01-08,2026-01-08::1301_CFFA6,1301_CFFA6,1301,CFFA6,0,0,0,0,0.0,...,10000000,1000000,1000000,35000000,1000000,0,0,numba,265,264
3,2026-01-08,2026-01-08::2409_CHFA6,2409_CHFA6,2409,CHFA6,0,0,0,0,0.0,...,10000000,1000000,1000000,35000000,1000000,0,0,numba,265,264
4,2026-01-08,2026-01-08::1605_CSFA6,1605_CSFA6,1605,CSFA6,0,0,0,0,0.0,...,10000000,1000000,1000000,35000000,1000000,0,0,numba,265,264
5,2026-01-08,2026-01-08::2408_CYFA6,2408_CYFA6,2408,CYFA6,0,0,0,0,0.0,...,10000000,1000000,1000000,35000000,1000000,0,0,numba,265,264
6,2026-01-08,2026-01-08::2603_CZFA6,2603_CZFA6,2603,CZFA6,0,0,0,0,0.0,...,10000000,1000000,1000000,35000000,1000000,0,0,numba,265,264
7,2026-01-08,2026-01-08::2609_DAFA6,2609_DAFA6,2609,DAFA6,0,0,0,0,0.0,...,10000000,1000000,1000000,35000000,1000000,0,0,numba,265,264
8,2026-01-08,2026-01-08::1101_DFFA6,1101_DFFA6,1101,DFFA6,0,0,0,0,0.0,...,10000000,1000000,1000000,35000000,1000000,0,0,numba,265,264
9,2026-01-08,2026-01-08::1326_DGFA6,1326_DGFA6,1326,DGFA6,0,0,0,0,0.0,...,10000000,1000000,1000000,35000000,1000000,0,0,numba,265,264


,trade_date,run_key,pair_name,rows,entry_signal_rows,entry_execution_rows,exit_execution_rows
0,2026-01-08,2026-01-08::1101_DFFA6,1101_DFFA6,0,0,0,0
1,2026-01-08,2026-01-08::1301_CFFA6,1301_CFFA6,0,0,0,0
2,2026-01-08,2026-01-08::1303_CAFA6,1303_CAFA6,0,0,0,0
3,2026-01-08,2026-01-08::1326_DGFA6,1326_DGFA6,0,0,0,0
4,2026-01-08,2026-01-08::1605_CSFA6,1605_CSFA6,0,0,0,0
5,2026-01-08,2026-01-08::1717_QOFA6,1717_QOFA6,0,0,0,0
6,2026-01-08,2026-01-08::1802_KUFA6,1802_KUFA6,188,94,84,10
7,2026-01-08,2026-01-08::2002_CBFA6,2002_CBFA6,0,0,0,0
8,2026-01-08,2026-01-08::2027_FEFA6,2027_FEFA6,0,0,0,0
9,2026-01-08,2026-01-08::2312_FSFA6,2312_FSFA6,0,0,0,0


,trade_date,run_key,pair_name,spot_symbol,future_symbol,universe_source,carried_from_date,initial_quantity,initial_direction,final_quantity,final_direction,expiry_date,is_expiry_date,carry_to_next_date,status
68,2026-01-09,2026-01-09::6770_QZFA6,6770_QZFA6,6770,QZFA6,carried_position,2026-01-08,9,ENTER_LONG_SPOT_SHORT_FUTURE,10,ENTER_LONG_SPOT_SHORT_FUTURE,2026-01-21,False,True,carried_forward
94,2026-01-12,2026-01-12::6770_QZFA6,6770_QZFA6,6770,QZFA6,carried_position,2026-01-09,10,ENTER_LONG_SPOT_SHORT_FUTURE,10,ENTER_LONG_SPOT_SHORT_FUTURE,2026-01-21,False,True,carried_forward
116,2026-01-13,2026-01-13::6282_KFFA6,6282_KFFA6,6282,KFFA6,selected+carried,2026-01-12,8,ENTER_LONG_SPOT_SHORT_FUTURE,0,HOLD,2026-01-21,False,False,closed
130,2026-01-13,2026-01-13::6770_QZFA6,6770_QZFA6,6770,QZFA6,carried_position,2026-01-12,10,ENTER_LONG_SPOT_SHORT_FUTURE,10,ENTER_LONG_SPOT_SHORT_FUTURE,2026-01-21,False,True,carried_forward
170,2026-01-14,2026-01-14::6770_QZFA6,6770_QZFA6,6770,QZFA6,carried_position,2026-01-13,10,ENTER_LONG_SPOT_SHORT_FUTURE,10,ENTER_LONG_SPOT_SHORT_FUTURE,2026-01-21,False,True,carried_forward
202,2026-01-15,2026-01-15::6770_QZFA6,6770_QZFA6,6770,QZFA6,carried_position,2026-01-14,10,ENTER_LONG_SPOT_SHORT_FUTURE,10,ENTER_LONG_SPOT_SHORT_FUTURE,2026-01-21,False,True,carried_forward
237,2026-01-16,2026-01-16::6770_QZFA6,6770_QZFA6,6770,QZFA6,carried_position,2026-01-15,10,ENTER_LONG_SPOT_SHORT_FUTURE,10,ENTER_LONG_SPOT_SHORT_FUTURE,2026-01-21,False,True,carried_forward
238,2026-01-19,2026-01-19::1303_CAFA6,1303_CAFA6,1303,CAFA6,selected+carried,2026-01-16,10,ENTER_LONG_SPOT_SHORT_FUTURE,0,HOLD,2026-01-21,False,False,closed
273,2026-01-19,2026-01-19::6770_QZFA6,6770_QZFA6,6770,QZFA6,carried_position,2026-01-16,10,ENTER_LONG_SPOT_SHORT_FUTURE,10,ENTER_LONG_SPOT_SHORT_FUTURE,2026-01-21,False,True,carried_forward
298,2026-01-20,2026-01-20::1717_QOFA6,1717_QOFA6,1717,QOFA6,selected+carried,2026-01-19,4,ENTER_LONG_SPOT_SHORT_FUTURE,0,HOLD,2026-01-21,False,False,closed


以下已確認的到期殘倉例外不納入本次稽核：


,trade_date,run_key,pair_name,spot_symbol,future_symbol,universe_source,carried_from_date,initial_quantity,initial_direction,final_quantity,final_direction,expiry_date,is_expiry_date,carry_to_next_date,status
1957,2026-04-15,2026-04-15::1802_KUFD6,1802_KUFD6,1802,KUFD6,selected,NaN,0,HOLD,10,ENTER_LONG_SPOT_SHORT_FUTURE,2026-04-15,True,False,expiry_position_remaining
4429,2026-07-15,2026-07-15::2408_CYFG6,2408_CYFG6,2408,CYFG6,selected,NaN,0,HOLD,1,ENTER_LONG_SPOT_SHORT_FUTURE,2026-07-15,True,False,expiry_position_remaining
4456,2026-07-15,2026-07-15::5371_NMFG6,5371_NMFG6,5371,NMFG6,selected+carried,2026-07-14,1,ENTER_LONG_SPOT_SHORT_FUTURE,1,ENTER_LONG_SPOT_SHORT_FUTURE,2026-07-15,True,False,expiry_position_remaining
4467,2026-07-15,2026-07-15::2340_FYFG6,2340_FYFG6,2340,FYFG6,carried_position,2026-07-14,3,ENTER_LONG_SPOT_SHORT_FUTURE,3,ENTER_LONG_SPOT_SHORT_FUTURE,2026-07-15,True,False,expiry_position_remaining
4468,2026-07-15,2026-07-15::6257_MQFG6,6257_MQFG6,6257,MQFG6,carried_position,2026-07-14,10,ENTER_LONG_SPOT_SHORT_FUTURE,10,ENTER_LONG_SPOT_SHORT_FUTURE,2026-07-15,True,False,expiry_position_remaining


跨日留倉稽核通過：未發現未列入例外清單的到期日殘倉。


## 3. 產生資金與績效報表

預設使用低記憶體 summary 模式：先釋放大型 in-memory frames，再以 25,000 rows/batch 從既有 CSV 彙總；不會重跑 HBT。

In [ ]:
reports = build_report_tables(artifacts)
display(reports.frame('symbol_profit').head(30))
display(reports.frame('failure_overview'))
display(reports.frame('roi_summary_including_open'))
display(reports.frame('capital_constraint_summary'))
display(reports.frame('daily_capital_constraint').tail(20))
print(f'Report CSV directory: {reports.output_dir}')

## 4. 儲存 PNG 圖表

In [ ]:
figure_paths = save_report_plots(artifacts, reports)
for name, path in figure_paths.items():
    print(f'{name}: {path}')
    display(Image(filename=str(path)))

## 5. 選擇性逐 pair 檢查

In [ ]:
selected_pair = reports.selected_pair
entry_exit = artifacts.frame('entry_exit_all')
latency = artifacts.frame('latency')
if selected_pair is None:
    print('No pair is available for drill-down.')
else:
    if 'pair_name' in entry_exit.columns:
        display(entry_exit.loc[entry_exit['pair_name'].eq(selected_pair)].head(100))
    if 'pair_name' in latency.columns:
        display(latency.loc[latency['pair_name'].eq(selected_pair)].head(120))